# 🧠 Notebook 02 — Sequence Encoder (CoLES)
## Financial Representation Learning System

**Purpose:** Train a self-supervised customer encoder using CoLES (Contrastive Learning for Event Sequences). No labels required.

**Training objective:** Two random subsequences of the same customer's history should produce similar embeddings. Different customers should be distant.

**Output:** `data/synthetic/customer_embeddings.pkl` — one 64-dim vector per customer

**Runtime:** ~12 minutes on Colab T4 GPU

⚡ **Enable GPU:** Runtime → Change runtime type → T4 GPU

## Setup

In [ ]:
import os, sys
os.chdir("/content/FRL-SYSTEM")
sys.path.insert(0, ".")

import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.encoder import (
    CustomerEncoder, CoLESDataset, train_encoder,
    encode_all_customers, save_encoder, save_embeddings
)
from src.config import ENCODER_OUTPUT_DIM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
if str(device) == "cpu":
    print("⚠️  No GPU detected. Training will be slower. Enable T4 in Runtime settings.")

## Step 1 — Load Sequences

In [ ]:
with open("data/synthetic/customer_sequences.pkl", "rb") as f:
    sequences = pickle.load(f)

print(f"Loaded {len(sequences):,} customer sequences")
sample = list(sequences.values())[0]
print(f"Token shape per customer: {sample['tokens'].shape}")
print(f"Features per token: {sample['tokens'].shape[1]}")

## Step 2 — Train the Encoder

In [ ]:
# Train CoLES encoder
# This may take 10-15 minutes on CPU, ~5 minutes on T4 GPU
model, history = train_encoder(sequences, device=device)

## Step 3 — Training Loss Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history, color='#3498DB', linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("NT-Xent Loss")
plt.title("CoLES Training Loss (lower = better separated embeddings)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("docs/results/02_training_loss.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\nFinal loss: {history[-1]:.4f}")

## Step 4 — Encode All Customers

In [ ]:
embeddings = encode_all_customers(model, sequences, device=device)

# Save
save_encoder(model, "data/synthetic/encoder.pt")
save_embeddings(embeddings, "data/synthetic/customer_embeddings.pkl")
print("\n✅ Saved encoder.pt and customer_embeddings.pkl")

## Step 5 — Quick Sanity Check

In [ ]:
# Check cosine similarity within vs across archetypes
emb_matrix = np.array([v["embedding"] for v in embeddings.values()])
archetypes = [v["archetype"] for v in embeddings.values()]

# Sample 200 customers
idx = np.random.choice(len(emb_matrix), 200, replace=False)
sample_embs = emb_matrix[idx]
sample_archs = [archetypes[i] for i in idx]

# Cosine similarity matrix
sim_matrix = sample_embs @ sample_embs.T  # already L2-normalised

# Within-archetype vs across-archetype similarity
within_sims, across_sims = [], []
arch_arr = np.array(sample_archs)
for i in range(len(sample_embs)):
    for j in range(i+1, len(sample_embs)):
        sim = sim_matrix[i, j]
        if arch_arr[i] == arch_arr[j]:
            within_sims.append(sim)
        else:
            across_sims.append(sim)

print(f"Within-archetype cosine similarity:  {np.mean(within_sims):.4f} (should be HIGHER)")
print(f"Across-archetype cosine similarity:  {np.mean(across_sims):.4f} (should be LOWER)")
print(f"\nSeparation ratio: {np.mean(within_sims)/max(np.mean(across_sims),0.001):.2f}x")
print("\n✅ If within > across, the encoder is learning meaningful representations")

## ✅ Notebook 02 Complete

**What was learned:**
- 64-dimensional customer embedding per customer
- Learned purely from event sequence structure (no labels)
- Within-archetype similarity > across-archetype similarity = meaningful representations

**Next:** Run `03_graph_builder.ipynb` to build the customer–merchant network.